# 6. Gain-TV-guided flagging: the experiment and its verdict

This is the first of the project's two research questions, told from the
question to the verdict. Notebook 5 teaches the campaign's machinery on the
fixture; this one follows the experiment through every pre-registered
campaign on real MeerKAT data and ends where the record ends: the question is
**closed**, and what the archives say about it is printed below, not retyped.

How to read it:

- cells marked **live** compute on the public fixture (four antennas, one
  pointing scan) while you watch; they are teaching aids, not results;
- cells marked **archive** read the frozen records in `research/results/`;
  every real-data number on this page comes from one of them;
- the optional replay of task K on a local LUNO slice runs only with
  `RANG_NOTEBOOK_LUNO=1` and prints its skip line otherwise.

The full write-up is `research/flagging-verdict.md` (task L4).

In [ ]:
%matplotlib inline
import os

os.environ.setdefault(
    "RAYON_NUM_THREADS", "4"
)  # share the cores; set before the first core call

import pandas as pd
from IPython.display import Markdown, display
from rang.notebook import data, experiment, flags, gains, science

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)

CACHE = data.default_cache_dir()  # choose your own with RANG_NOTEBOOK_CACHE
LUNO = science.switch("RANG_NOTEBOOK_LUNO")
print("cache:", CACHE)
print("LUNO replay:", "on" if LUNO else "off")

## 1. The question

Calibration gains of a well-behaved telescope vary slowly in time and
frequency. Radio-frequency interference (RFI) that leaks into a solve makes
them jump. **Can the roughness of the gains tell a flagger where the bad data
are?**

The method, frozen on 2026-09-23 before any real data were seen:

1. Solve full-Jones gains $G$ per solution bin (QuartiCal's `complex` term,
   ported to the core).
2. Fix the common unitary gauge on a reference antenna and map the
   anisotropic Frobenius total variation (TV) of the gains per antenna, time
   bin and frequency bin: the guidance $h$.
3. Propose a training cell when its robust residual score $z$ is an outlier,
   lowering the threshold where the gains are rough:

$$z \geq 2 \quad\text{and}\quad z\,[1+\alpha\min(h/\operatorname{median}h,\,5)]>\tau .$$

   The unguided control is $\alpha=0$. At most a fixed fraction of training
   cells may be added, and original flags are never removed.
4. Re-solve every distinct candidate mask $F$ and score it against the
   original flags $F_0$:

$$L(F)=\frac{\mathrm{TV}(F)}{\mathrm{TV}(F_0)}+w_f\,f+w_v\,\frac{\mathrm{MSE}_{\mathrm{val}}(F)}{\mathrm{MSE}_{\mathrm{val}}(F_0)},$$

   with $f$ the added fraction; a candidate is admissible only if its
   validation MSE stays within the tolerance of the original. "No change"
   is always a candidate. Test cells never choose anything.

The frozen values, read from `rang.core_campaign`:

In [ ]:
pd.Series(science.flagging_rules(), name="frozen value").to_frame()

**The decision rule** (frozen per campaign before each run, quoted from
`rang.notebook.experiment.DECISION_RULE`):

In [ ]:
display(Markdown("> " + experiment.DECISION_RULE))
print(
    "cost clause limit (task K's archive): TV chain within",
    science.cost_limit(),
    "x the first pass",
)

A **positive** result was therefore a *better* headline call, the TV policy
against CARACAL's first-pass Tricolour strategy, with the cost clause met. The
project's novelty rule (`AGENTS.md`) asks for higher quality at comparable
cost, or comparable quality at materially lower cost.

## 2. The method, live on the fixture (exploratory)

The fixture is scan 9 of the public SKA pointing-offset archive: four
MeerKAT antennas on PKS 1934−638, real flags. First the guidance itself: the
TV map of the original solve. Bins without data are grey.

In [ ]:
ds = data.load_fixture(cache_dir=CACHE)
original = gains.solve(ds)  # frozen grid and iteration cap
display(gains.losses_table({"original": original}))
gains.gain_maps(ds, original, "tv")

Now the campaign structure: the original arm, a comparator arm with a
Tricolour strategy's additions (CARACAL's first pass when its strategy files
are found, the teaching strategy otherwise), and the frozen TV policy loop.
**What to look for:** which candidate masks are admissible, and whether any
TV-guided mask (tv weight > 0) scores below the no-change objective.

In [ ]:
library = flags.strategy_library()
comparator = (
    "caracal-calibrator-firstpass"
    if "caracal-calibrator-firstpass" in library
    else "teaching"
)
arms, trials = experiment.run_arms(ds, library[comparator])
images = experiment.image_arms(ds, arms)
print("comparator strategy:", comparator)
display(experiment.arms_table(arms, images))
experiment.trials_table(trials)

In [ ]:
calls = experiment.decisions(arms, images)
display(calls)
table = experiment.trials_table(trials)
guided = table[(table["tv weight"] > 0) & table["accepted"].fillna(False).astype(bool)]
display(
    Markdown(
        f"On the fixture the loop tried **{len(table)}** candidates; "
        f"**{len(guided)}** TV-guided masks were accepted, and the policy arm added "
        f"**{int(arms['tv-policy']['added'].sum())}** cells. This replay is a teaching "
        "aid on a pointing scan, not a pre-registered result."
    )
)
experiment.ratio_figure(calls, f"fixture replay (exploratory), comparator {comparator}")

**Reading the replay honestly.** When the policy selects no flag, as the cell
above reports, the policy arm equals the original arm. CARACAL's first pass,
when its strategy files are found, removes so many training cells on the
fixture that some held-out cells lose their gain solution: the test MSE is
then *undecidable*, not better or worse, and a residual-RMS call in the
policy's favour mostly says that the comparator imaged fewer cells (notebook 5
shows this in detail). The documentation build has no strategy files and uses
the teaching strategy, so its replay differs from one with CARACAL's. Either
way this is a pointing scan, not the experiment.

## 3. What the real data said

The real test used the public LUNO MeerKAT L-band observation (two
calibrators, processed flags and weights), with every stage on the Rust
core. Each wave 3 protocol answered one part of the question and left one
archive:

In [ ]:
science.wave3_table()

Task L4 collected every pre-registered call from those archives into one
table, `research/results/flagging-verdict.json`, and
`examples/verdict_tables.py --check` re-derives each call from its ratio and
fails if the table is stale. Run it (**live**, standard library only):

In [ ]:
check = science.verdict_tables_check()
print("verdict table check:", "not in a checkout" if check is None else check[1])
record = science.verdict()
print(record["task"])
print("rule:", record["rule"])

### The calls (archive)

In [ ]:
display(Markdown(science.verdict_summary_markdown(record)))
science.call_counts(record)

In [ ]:
science.calls_figure(record, "every pre-registered call against the frozen null band")

**Why so many undecidable calls.** On the whole array the first pass removes
every training sample of some station-bins; the solver then flags those
gains and the held-out cells there cannot be predicted, so the frozen
test-MSE call has no value. Two pre-registered rules followed, each committed
before the run it governs: *amendment 1* compares a pair on the held-out
samples both arms predict, and the *A1 drop* removes the held-out samples of
the station-bins the Tricolour strategies empty (GNSS-band residue, section
4). The `rule` column says which rule made each call. The headline pairs, the
rows a positive would have had to come from:

In [ ]:
science.headline_calls(record)

### What the TV policy chose (archive)

`selected_policy` is the mask the objective preferred; `guided_accepted`
counts accepted masks with $\alpha>0$, the TV-guided ones.

In [ ]:
science.policy_selections(record)

### The cost clause (archive)

The TV chain is a reference solve plus one re-solve per distinct candidate
mask. Kernel changes during the project (parallel Tricolour, parallel
solves) moved the ratio, so each run states its kernels and host load.

In [ ]:
display(science.cost_table(record))
science.cost_figure(record)

## 4. Was something else the cause? (archive)

Four follow-up protocols asked whether the null came from something other
than the idea itself: the calibrator model (L1-M), structure that gains
cannot absorb (C1), an unconverged solver (L5) and the emptied GNSS bins
(A1). CARACAL's own CASA cross-calibration (O-CASA) is the matched
calibration baseline on four stations.

In [ ]:
ctx = science.context_tables(record)
for title, frame in ctx.items():
    display(Markdown(f"**{title}**"))
    display(frame)

In [ ]:
raw = record["context"]
l1m = {(c["pair"], c["metric"]): c for c in raw["l1m_model_calls"]}
fs = l1m[("field vs stored", "original_mse_test")]
fp = l1m[("field vs point", "original_mse_test")]
c1 = raw["c1_q1_trace"]
l5 = raw["l5_array"]
display(
    Markdown(
        f"- **L1-M.** CARACAL's field model against the stored model: held-out MSE ratio "
        f"{fs['ratio']:.4f} ({fs['call']}); against a point model {fp['ratio']:.4f} "
        f"({fp['call']}). The model moves held-out error; the flagging decisions did not move "
        f"(the L1-M rows of the call table above).\n"
        f"- **C1.** The closure-TV map is reproducible (split-half ρ {c1['rho_half']:.3f}) and "
        f"the gain-TV map explains R² = {c1['r2_bl']:.4f} of it: call *{c1['call']}*. "
        f"Used as guidance or as an outlier mask it changed no call.\n"
        f"- **L5.** At the {science.flagging_rules()['solver iterations']}-iteration cap {l5['D']['converged_fraction']:.1%} of the array's "
        f"gains pass the convergence test; Anderson acceleration (m = 3) reaches "
        f"{l5['anderson-3']['converged_fraction']:.1%} in {l5['anderson-3']['iterations']} "
        f"iterations with test MSE {l5['anderson-3']['losses']['test']:.7f} against "
        f"{l5['D']['losses']['test']:.7f}. The flagging loop was not rerun on converged gains.\n"
        f"- **A1.** The station-bins the first pass empties ({raw['a1_emptied']['l1']} on "
        f"1934−638, {raw['a1_emptied']['l2']} on 1830−211) are GNSS-band residue: RFI "
        "excision as intended, so a method earns no credit for keeping them."
    )
)

**The fixture era** (2026-09, QuartiCal in a container; archive): the frozen
policy selected nothing on the fixture; the only accepted non-empty mask came
from a later ranking search on an injected burst, where the TV ranking and
the plain residual ranking chose the same cells.

In [ ]:
pd.Series(science.fixture_era(record), name="fixture era").to_frame()

## 5. What the null does and does not say

- It **does** say: on one processed MeerKAT L-band observation (both
  calibrators, four stations and the whole array, several calibrator models,
  a gain statistic and a gauge-free closure statistic), gain-TV guidance
  selected no flag that improves held-out prediction or the image residual
  over CARACAL's first pass, and it is not cheaper. Lower gain TV did not mean
  better prediction.
- It **does not** cover native, unprocessed flags and weights; a science
  target rather than a calibrator; another observation, band or telescope; or
  a closure-based or TV-*regularised* objective. The first would reopen this
  experiment; the others would be new experiments.

**What would reopen it: L3, native data.** The observer offered a
full-resolution L-band set with its original FLAG, WEIGHT_SPECTRUM and
SIGMA_SPECTRUM; nothing has arrived, so L3 never ran. When it arrives, the
frozen campaign runs on it unchanged, with the decision rule above. Section 7
of `research/flagging-verdict.md` lists the other items (correctly scaled
weights, a converged solver, solution-interval and regularisation choices, a
closure objective, a target field) with what speaks for and against each.

## 6. Replay task K from a local slice (optional)

With `RANG_NOTEBOOK_LUNO=1`, a local copy of the four-station LUNO slice and
CARACAL's strategy files, the cell below reruns task K's arms and must give
its archived calls exactly (the core is deterministic).

In [ ]:
if not LUNO:
    print(science.skip_line("RANG_NOTEBOOK_LUNO"))
elif data.find_luno_slice() is None:
    print(science.skip_line("RANG_NOTEBOOK_LUNO", "no local LUNO slice found"))
elif "caracal-calibrator-firstpass" not in library:
    print(
        science.skip_line("RANG_NOTEBOOK_LUNO", "CARACAL first-pass strategy not found")
    )
else:
    luno = data.load_luno()
    k_arms, _ = experiment.run_arms(luno, library["caracal-calibrator-firstpass"])
    k_images = experiment.image_arms(luno, k_arms)
    replay = experiment.decisions(k_arms, k_images)
    archived = science.archived_hypotheses("core-k-luno-campaign.json")
    display(replay)
    mine = replay[
        (replay.candidate == "tv-policy") & (replay.reference == "comparator")
    ]
    frozen = archived[
        (archived.candidate == "tv-policy")
        & (archived.reference == "tricolour-firstpass")
    ]
    print("headline test-MSE ratio, replay: ", mine["mse_test ratio"].iloc[0])
    print("headline test-MSE ratio, archive:", frozen["mse_test ratio"].iloc[0])

---

Next: notebook 7 takes up the project's original question, whether small
pointing and beam corrections can be constrained on MeerKAT data, which the
project returned to when this one closed.